In [0]:
from pyspark.sql import functions as F

# ============================================================
# CELL 1 — Carregar usando eval_set corretamente
# ============================================================
orders = spark.table("big_data.silver.orders")  # tem coluna eval_set
order_products = spark.table("big_data.silver.order_products")

prior_orders = orders.filter(F.col("eval_set") == "prior")
train_orders = orders.filter(F.col("eval_set") == "train")
test_orders  = orders.filter(F.col("eval_set") == "test")

# Pedidos de treino do modelo = prior + train (comportamento histórico completo)
train_full = prior_orders.union(train_orders)

# Transações para FP-Growth / CF: apenas prior
transactions_prior = (
    order_products.join(prior_orders, "order_id")
    .groupBy("user_id")
    .agg(F.flatten(F.collect_list("product_id")).alias("all_items"))
)

In [0]:
# ============================================================
# CELL 2 — Construir ground truth para avaliação
# ============================================================
# O target correto é: o que o usuário comprou no pedido "test"
# Foco em cross-sell genuíno: produtos que NÃO estavam no histórico prior

# Itens do histórico de cada user
prior_items_by_user = (
    order_products.join(prior_orders, "order_id")
    .groupBy("user_id")
    .agg(F.collect_set("product_id").alias("prior_items"))
)

# Itens do pedido de teste
test_items_by_user = (
    order_products.join(test_orders, "order_id")
    .groupBy("user_id")
    .agg(F.collect_set("product_id").alias("test_items"))
)

# Contexto: último estado do carrinho conhecido (pedido train)
train_items_by_user = (
    order_products.join(train_orders, "order_id")
    .groupBy("user_id")
    .agg(F.collect_list("product_id").alias("context_items"))
)

# Juntar tudo
eval_df = (
    train_items_by_user
    .join(test_items_by_user, "user_id")
    .join(prior_items_by_user, "user_id")
    .withColumn(
        # Cross-sell genuíno = items em test que NÃO aparecem em prior
        "new_items_in_test",
        F.array_except("test_items", "prior_items")
    )
)

# Para avaliação padrão: target = todos os itens do pedido test
# Para cross-sell específico: target = apenas new_items_in_test

In [0]:
import pandas as pd

# ============================================================
# CELL 3 — Avaliação com métricas corretas para Instacart
# ============================================================
# O benchmark público do Instacart usa F1@K como métrica principal
# F1 médio de ~0.36–0.40 é o estado da arte

def evaluate_user(context, targets_set, sim_df, k=5):
    """Avalia 1 usuário. targets_set é um set de produtos esperados."""
    recs = recommend_cf(context, sim_df, k=k)  # lista de product_ids
    
    hits = len(set(recs) & targets_set)
    precision = hits / k if k > 0 else 0
    recall    = hits / len(targets_set) if targets_set else 0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0
    
    return precision, recall, f1

# Avaliar para todos os usuários no conjunto de avaliação
results = []
for row in eval_df.limit(1000).collect():
    ctx     = row["context_items"]
    targets = set(row["test_items"])   # ou "new_items_in_test" para cross-sell puro
    
    if not ctx or not targets:
        continue
    
    p, r, f1 = evaluate_user(ctx, targets, similarity_matrix, k=5)
    results.append({"precision": p, "recall": r, "f1": f1})

df_res = pd.DataFrame(results)

if df_res.empty:
    print("❌ ERRO: Nenhum resultado gerado!")
    print("\nProblemas identificados:")
    print("1. eval_df está vazio (0 linhas)")
    print("   - Não há pedidos com eval_set='test' nos dados")
    print("   - Valores disponíveis: 'prior' e 'train'")
    print("\n2. Funções não definidas:")
    print("   - recommend_cf() precisa ser implementada")
    print("   - similarity_matrix precisa ser criada")
    print("\nSoluções:")
    print("- Use 'train' como conjunto de teste (padrão Instacart)")
    print("- Construa a matriz de similaridade com FP-Growth ou item-item CF")
    print("- Implemente a função recommend_cf para gerar recomendações")
else:
    print(f"Precision@5 médio : {df_res['precision'].mean():.4f}")
    print(f"Recall@5 médio    : {df_res['recall'].mean():.4f}")
    print(f"F1@5 médio        : {df_res['f1'].mean():.4f}")
    print(f"\nBenchmark Instacart (estado da arte): F1 ~ 0.36–0.40")